# Notebook 02 — A/B Test

## Business Question
Did the pricing pilot cause a statistically significant revenue lift in the three treated departments, or could the observed difference be random chance?

## Method
Classic two-sample t-test treating stores as the unit of analysis.
Pilot stores (n=8) = treatment group. Control stores (n=11) = control group.

## Structure
1. **A/A Test** — split control stores into two halves, verify no significant difference in pre-period revenue. Validates the t-test framework before using it.
2. **Power Analysis** — given n=8 pilot stores, MDE=5%, power=80%, what is the smallest effect we could reliably detect?
3. **A/B Test** — one t-test per pilot department during Apr–Jun 2024. Is the lift statistically significant?
4. **Summary** — results table with lift %, p-value, and significance flag per department.

## Key Assumptions
- Stores are independent units (no spillover between stores — SUTVA holds)
- Revenue is approximately normally distributed within each group
- Store 9 (online) excluded throughout

## Limitation
T-test does not control for pre-period differences between stores or time trends.

## Data
Input: `data/analytical_table.csv` (built in data_prep)

# Load Data

In [1]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

ImportError: DLL load failed while importing _min_spanning_tree: An Application Control policy has blocked this file.

In [ ]:
analytical_table = pd.read_csv('../data/analytical_table.csv',parse_dates=['week_start_date'])

In [ ]:
analytical_table.info()

In [ ]:
analytical_table.head()

In [ ]:
# Exclude Store 9 - online pricing
df = analytical_table[analytical_table['store_id'] != 9].copy()

In [ ]:
# Defne basiline period and pilot periods

PILOT_START = pd.Timestamp("2024-04-01")
PILOT_END   = pd.Timestamp("2024-06-30")
PRE_START   = pd.Timestamp("2023-04-01")
PRE_END     = pd.Timestamp("2024-03-31")

print(f"Shape: {df.shape}")
print(f"Pilot stores:   {df[df['is_pilot']==True]['store_id'].nunique()}")
print(f"Control stores: {df[df['is_pilot']==False]['store_id'].nunique()}")

# A/A Test
**Split control stores into two random halves and compare their**

**pre-period revenue. Expect p > 0.05 — no significant difference.**

**If this fails, the control group is broken before we even start.**

In [ ]:
pre_period  = df[(df['week_start_date'] >= PRE_START) & (df['week_start_date'] <= PRE_END) & (df['is_pilot'] == False) ]

# Aggregate to weekly revenue per store (all depts combined)
control_weekly = (pre_period.groupby(['store_id', 'week_start_date'])['weekly_revenue'].sum().reset_index())

# Split control stores into two random halves
np.random.seed(42)
control_store_ids = control_weekly['store_id'].unique()
half              = len(control_store_ids) // 2
group_a_stores    = np.random.choice(control_store_ids, size=half, replace=False)
group_b_stores    = np.array([s for s in control_store_ids if s not in group_a_stores])


In [ ]:
print(group_a_stores)
print(group_b_stores)

In [ ]:
# Extract revenue and run A/A t-test
group_a_rev = control_weekly[control_weekly['store_id'].isin(group_a_stores)]['weekly_revenue']
group_b_rev = control_weekly[control_weekly['store_id'].isin(group_b_stores)]['weekly_revenue']

t_stat, p_val = ttest_ind(group_a_rev, group_b_rev)


print("=" * 55)
print("A/A TEST RESULTS")
print("=" * 55)
print(f"Group A stores:  {sorted(group_a_stores)}")
print(f"Group B stores:  {sorted(group_b_stores)}")
print(f"Group A mean:    CAD {group_a_rev.mean():,.0f}")
print(f"Group B mean:    CAD {group_b_rev.mean():,.0f}")
print(f"t-statistic:     {t_stat:.4f}")
print(f"p-value:         {p_val:.4f}")
print(f"Result: {'✅ PASS — control group is clean (p > 0.05)' if p_val > 0.05 else ' FAIL — significant difference detected'}")

### A/A Test Result — FAIL (p = 0.0016)

The A/A test failed — Group A (CAD 175,182) and Group B (CAD 194,942) show a
statistically significant difference despite both being control stores.

**Root cause:** Small sample size (n=11 control stores). Random 5/6 split by
chance assigned larger stores to Group B and smaller stores to Group A.
With only 11 stores, random halving cannot guarantee size-balanced groups.
This is a sample size limitation.

**Implication for A/B test:** Do not use random store assignment.
Instead use the matched pairs from Notebook 01 — each pilot store is compared
only to its most similar control store, controlling for size and revenue baseline.

**In production:** With hundreds of stores, propensity score matching via
logistic regression would be used. At n=20 stores, Euclidean distance
matching on z-scored features is the appropriate method.

# A/B Test

## Power Analysis

In [ ]:
# ── Power Analysis ───────────────────────────────────────────────────
# How small an effect can we reliably detect with n=8 pilot stores?
# If MDE > true effect, insignificant A/B results are expected.



n_pilot, n_control = 8, 11
alpha, power = 0.05, 0.80

# Calculate minimum detectable effect size (Cohen's d)
z_alpha = norm.ppf(1 - alpha/2)  # 1.96
z_beta  = norm.ppf(power)         # 0.84
mde_d   = (z_alpha + z_beta) * np.sqrt(1/n_pilot + 1/n_control)

# Convert Cohen's d to % using pre-period revenue baseline
pre_revenue = df[
    (df['week_start_date'] >= PRE_START) &
    (df['week_start_date'] <= PRE_END)
].groupby(['store_id', 'week_start_date'])['weekly_revenue'].sum()

mde_pct = (mde_d * pre_revenue.std() / pre_revenue.mean()) * 100

print("=" * 50)
print("POWER ANALYSIS")
print("=" * 50)
print(f"Pilot stores (n):      {n_pilot}")
print(f"Control stores (n):    {n_control}")
print(f"Alpha:                 {alpha}")
print(f"Power:                 {power}")
print(f"MDE (Cohen's d):       {mde_d:.3f}")
print(f"MDE as % lift:         {mde_pct:.1f}%")
print(f"Industry threshold:    5%")
print(f"\nResult: {'Adequately powered' if mde_pct <= 5 else f' Underpowered — MDE is {mde_pct:.1f}%, above 5% threshold'}")

## A/B Test

In [ ]:
# ── A/B Test using matched pairs ─────────────────────────────────────
# Use matched_control_id from stores to compare each pilot store
# only against its closest control store — eliminates size imbalance

pilot_period = df[
    (df['week_start_date'] >= PILOT_START) &
    (df['week_start_date'] <= PILOT_END)
]

# Get matched control store IDs for pilot stores
matched_controls = df[df['is_pilot'] == True][['store_id', 'matched_control_id']].drop_duplicates()
matched_control_ids = matched_controls['matched_control_id'].dropna().astype(int).tolist()

PILOT_DEPTS = {1: 'Electronics', 2: 'Home & Kitchen', 3: 'Sports & Outdoors'}

results = []
print("=" * 60)
print("A/B TEST — Matched Pairs (Pilot vs Matched Control)")
print("Pilot period: Apr–Jun 2024")
print("=" * 60)

for dept_id, dept_name in PILOT_DEPTS.items():
    dept_data = pilot_period[pilot_period['dept_id'] == dept_id]

    pilot_rev   = dept_data[dept_data['is_pilot'] == True]['weekly_revenue']
    control_rev = dept_data[dept_data['store_id'].isin(matched_control_ids)]['weekly_revenue']
    
    t_stat, p_val = ttest_ind(pilot_rev, control_rev)
    lift_pct = ((pilot_rev.mean() - control_rev.mean()) / control_rev.mean()) * 100

    results.append({
        'Department':   dept_name,
        'Pilot Mean':   round(pilot_rev.mean(), 0),
        'Control Mean': round(control_rev.mean(), 0),
        'Lift %':       round(lift_pct, 2),
        'p-value':      round(p_val, 4),
        'Significant':  'Yes' if p_val < 0.05 else 'No'
    })

    print(f"\n{dept_name}")
    print(f"  Pilot mean:   CAD {pilot_rev.mean():,.0f}")
    print(f"  Control mean: CAD {control_rev.mean():,.0f}")
    print(f"  Lift:         {lift_pct:+.2f}%")
    print(f"  p-value:      {p_val:.4f}")
    print(f"  Significant:  {'Yes' if p_val < 0.05 else 'No'}")

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print(results_df.to_string(index=False))


### A/B Test Results — Key Insights

**All three departments show insignificant results (p > 0.05).** However this is
expected given the power analysis finding — the test is underpowered.

- **Electronics** showed the strongest directional signal: +9.32% lift (p=0.1171).
  Direction is correct — ground truth ATT is positive. Insignificance is due to
  small sample size.

- **Home & Kitchen** showed minimal lift: +0.77% (p=0.8910). Consistent with
  the small true ATT expected for medium-elasticity departments.

- **Sports & Outdoors** showed a negative lift: -3.84% (p=0.4852). Direction
  is correct — high elasticity department expected to lose revenue from price change.

**Why results are insignificant despite correct direction:**
With MDE = 56.1%, the t-test needs to see a 56% lift to call it significant.
The true effects (12%, 3%, -2%) are far below this threshold. This is a
fundamental limitation of small-sample A/B testing.

**Conclusion:** A/B test alone is insufficient at n=8 stores. The directionally
correct results motivate using more powerful causal methods — DiD ,
Synthetic Control , and BSTS — which control for
pre-period trends and store-level differences to extract signal from small samples.

**How to properly conduct an A/B test with only 8 pilot and 11 control stores:**

1. **Run the test longer** — extend pilot to 6–12 months. More observations
   per store reduce variance and lower the MDE.
2. **Use CUPED** — incorporate pre-period revenue as a covariate to reduce
   variance by 30–50% without adding stores. Industry standard at Netflix,
   Booking, and Microsoft.
3. **Switch to paired t-test** — compare each pilot store directly to its
   matched control store as a pair. More powerful than independent samples t-test.
4. **Use one-tailed test** — if business has a directional hypothesis,
   one-tailed test reduces MDE by approximately 20%.

**Other methods**
   - **Mann-Whitney U** — non-parametric alternative, does not assume normality,
  suitable when revenue distribution is skewed
- **Permutation test** — assumption-free, most appropriate for n=8 small samples,
  makes no distributional assumptions at all
- **Paired t-test** — compares each pilot store directly to its matched control
  store as a pair, more powerful than independent samples t-test at small n

T-test (independent samples) was chosen for interpretability and consistency
with industry standard A/B test reporting at store level.

## Some Notes:

## Understanding the A/B Test Results — Why Insignificant Despite Correct Direction

### What is the treatment?
The treatment is the price change applied by the pricing tool to 3 departments
(Electronics, Home & Kitchen, Sports & Outdoors) in 8 pilot stores during
Apr–Jun 2024. The treatment effect is the revenue lift or drop CAUSED by
that price change — not the total revenue, just the additional revenue
attributable to the new pricing.

### Key statistical concepts

**Standard Deviation** measures how spread out individual store revenues are
around the group average. In our data, pilot store weekly revenues range from
roughly CAD 15,000 to CAD 90,000 — a large spread driven by natural differences
in store size, location, and customer base, not by the treatment.

**Standard Error** measures how reliable our group average is as an estimate.
It equals standard deviation divided by the square root of sample size.
With only 8 pilot stores and high standard deviation between them, the
standard error is large — meaning our pilot group average is an unreliable
estimate with a wide margin of uncertainty.

**The t-test** compares the difference between pilot and control group averages
against the standard error. If the difference is large relative to the standard
error, the result is statistically significant. If the standard error is large
relative to the difference, the result is insignificant — the signal gets
buried in the noise.

### Why the A/B test was insignificant

The true treatment effects are modest: +12% for Electronics, +3% for Home
& Kitchen, -2% for Sports & Outdoors. For a store averaging CAD 50,000/week
in Electronics, a 12% lift is roughly CAD 6,000/week.

But the natural variation between stores is roughly CAD 75,000 (difference
between the smallest and largest pilot store). We are trying to detect a
CAD 6,000 signal inside CAD 75,000 of noise. The signal exists but the
noise drowns it out statistically.

Power analysis confirmed this: with n=8 pilot stores, the minimum detectable
effect (MDE) is 56.1% — meaning the test can only reliably detect a lift
above 56%. Our true effects (3–12%) are far below this threshold. The
insignificant results were mathematically expected before the test was run.

### Why matching did not fully solve this

Store matching made the pilot-vs-control comparison fairer by pairing
each pilot store with a similar control store (matched on size, footfall,
and baseline revenue). However, matching does not reduce the variance
WITHIN the pilot group itself — the 8 pilot stores still range from small
to large, which keeps the standard error high.

### How this would be handled in production

1. **More stores** — Walmart (400+ stores) or Loblaw (500+ stores) can
   run pilots with n=40+ per group, reducing MDE to detectable levels
2. **Longer pilot periods** — 6–12 months instead of 13 weeks gives more
   data points per store, reducing variance in the weekly average
3. **SKU-level analysis** — compare repriced vs non-repriced products
   within the SAME store, eliminating store-level noise entirely.
   This is the industry standard at Staples, Amazon, and Walmart
4. **CUPED** — use pre-period revenue as a covariate to reduce variance
   by 30–50% without adding more stores. Industry standard at Netflix,
   Booking, and Microsoft

### What the A/B test tells us despite being insignificant

The directional results are correct — Electronics positive, Home & Kitchen
slightly positive, Sports & Outdoors negative. This validates the simulation
design and confirms the causal methods in Notebooks 03–05 are worth pursuing.
The A/B test is not wrong — it is simply underpowered. More sophisticated
methods (DiD, Synthetic Control, BSTS) control for pre-period trends and
time effects, extracting signal that the t-test cannot.